# logit-thermo 探索 8.40（PROJECT_G の下見）：くり返しの罰を「どこにかけるか」で、同じ副作用あたりの効き目は変わるか——ランタイム 1/3（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。
- **3 本のノートブック（ランタイム 1/3・2/3・3/3）を、別々のランタイムで同時に回してください。** 30 個の仕事（3 モデル × 10 組：罰なし、いつも・芽を摘む・溶かす × 強さ 3 段）を、3 つのランタイムが「まだ終わっていない・誰も取っていない」ものから取りに行きます。3 つ同時なら 1 時間くらいで全部終わる見込みです。1 つだけでも、時間はかかりますが全部回せます。
- 作り：温度 0.5・0.6、記事 40 × サンプル 4、512 歩。罰 ＝ 写しの候補の logit に h ＝ −0.5・−1.0・−2.0 を足す。各歩の q、エネルギー U、副作用（KL）、罰をかけたかも記録します。モデル：Pythia-1B・OLMo-2-1B・Qwen2.5-1.5B。
- **データは、分析を固めて予想を出すまで開けません。** 結果は表示しません。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコード、`scripts/a2_explore_field.py`（`0ac42c7ab82b06d2…`）、`scripts/a2_explore_dsc.py`（`33f47a0a3bbda183…`）、`scripts/a2_explore_dsc2.py`（`cddae78690360418…`）は Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_explore_pork.py`（`9cabe53a42dbdabb…`）はこのノートブックに埋め込み。
- 保存先 `A2_explore/pork/`。途中で切れても、もう一度「すべて実行」すれば、終わった仕事は飛ばします（取りかけの仕事は 2 時間たつとほかのランタイムが取り直せます）。
- **通知**：🚀・仕事ごとの ✅・🎉（このランタイムで回した数と、まだ残っている仕事）・⚠️。名札は「logit-thermo 探索 PROJECT_G 罰の置き場所（3 モデル × 10 組、ランタイム 1/3、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 PROJECT_G 罰の置き場所（3 モデル × 10 組、ランタイム 1/3、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore PROJECT_G worker 1of3', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', 'SRCD2': 'cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF, SRCD = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF'], _s['SRCD']
SRCD, SRCD2 = _s['SRCD'], _s['SRCD2']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
SRCG = '"""探索 8.40（下見）：PROJECT_G ——くり返しの罰を「どこにかけるか」で、同じ副作用あたりの効き目は変わるか（2026-11-01 06:3x JST、GPU（L4）を 3 つ、判定しない）。\nscripts/a2_2_collect.py・a2_4_collect.py・a2_7_collect.py・a2_7b_collect.py・a2_explore_field.py を実行したあとの同じ名前空間で実行する\n（load_model・valid_mask・split_doc・CONT・CopyIndexN・B7_N を変えずに使う）。\n\n罰 ＝ 写しの候補（3 つ組の続き）の logit に h（負）を足す（8.30 の場と同じ）。1 本の文ごとに、続けて候補を選んだ歩の数 k（8.37 のはしご）を見て、かけるかを決める：\n  \'none\'   ：かけない\n  \'all\'    ：いつも（k によらず、候補があればかける）\n  \'grow\'   ：芽を摘む（1 ≤ k ≤ 7：写しを続けはじめた途中だけ。最初の 1 回の写しにはかけない）\n  \'melt\'   ：溶かす（k ≥ 8：結晶になってから）\n強さ h ＝ −0.5・−1.0・−2.0。温度 0.5・0.6。記事 40 × サンプル 4、512 歩。種は罰の組によらず同じ（共通の乱数）。\n副作用：罰をかけた歩の KL（罰の後の分布 ‖ 罰の前の分布、その温度で）。かけない歩は 0。\n記録：トークン、候補があったか、候補を選んだか、U（温度 1・罰なしの −ln p）、q0・q（罰の前・後の候補の確率の和）、KL、罰をかけたか。\n\n仕事の割り振り：30 個の仕事（3 モデル × 10 組）を、ランタイムが「まだ終わっていない・誰も取っていない」ものから取りに行く（G-pre と同じしくみ）。\n種は仕事だけで決まるので、2 つのランタイムが同じ仕事をしても結果は同じ。\n"""\nimport gc, os, time, json, socket\nimport numpy as np\nimport torch\n\nPK_TEMPS = [0.5, 0.6]\nPK_MODES = [\'all\', \'grow\', \'melt\']\nPK_H = [-0.5, -1.0, -2.0]\nPK_S = 4\nPK_SEED = 20261144\nPK_CHUNK = 20\nPK_STALE = 2 * 3600\nPK_KGROW = (1, 7)\nPK_KMELT = 8\nPK_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\')]\n\n\ndef pk_configs():\n    return [(\'none\', 0.0)] + [(m, h) for m in PK_MODES for h in PK_H]\n\n\ndef pk_tag(mode, h):\n    return \'none\' if mode == \'none\' else f\'{mode}_h{int(round(-h * 10)):02d}\'\n\n\ndef pk_seed(k, s, c):\n    return PK_SEED + 1000 * k + 10 * s + c\n\n\ndef pk_mask(mode, kk):\n    """kk：(B,) の続けて写した歩の数 → 罰をかけるか (B,)。"""\n    if mode == \'none\':\n        return torch.zeros_like(kk, dtype=torch.bool)\n    if mode == \'all\':\n        return torch.ones_like(kk, dtype=torch.bool)\n    if mode == \'grow\':\n        return (kk >= PK_KGROW[0]) & (kk <= PK_KGROW[1])\n    if mode == \'melt\':\n        return kk >= PK_KMELT\n    raise ValueError(mode)\n\n\n@torch.no_grad()\ndef generate_pork(model, prompts, T, h, mode, valid, gen, L=None):\n    """一定の温度。罰は 1 本ごとの k で決めた歩だけにかける。手順と乱数の使い方は generate_field・generate_dsc2 と同じ（罰がなければ同じトークン）。"""\n    L = CONT if L is None else L\n    device = valid.device\n    B = len(prompts)\n    x = torch.tensor(prompts, device=device)\n    out = model(x, use_cache=True)\n    past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    idx = [CopyIndexN(p, B7_N) for p in prompts]\n    kk = torch.zeros(B, dtype=torch.long, device=device)\n    rec = {k: [] for k in (\'toks\', \'cand\', \'pick\', \'U\', \'q0\', \'q\', \'kl\', \'app\')}\n    for t in range(L):\n        z = logits.double().masked_fill(~valid, float(\'-inf\'))\n        lp1 = torch.log_softmax(z, dim=-1)\n        cands = [idx[b].candidates() for b in range(B)]\n        has = torch.tensor([len(c) > 0 for c in cands], device=device)\n        rows, cols = [], []\n        for b, c in enumerate(cands):\n            rows += [b] * len(c); cols += list(c)\n        rt = torch.tensor(rows, device=device, dtype=torch.long); ct = torch.tensor(cols, device=device, dtype=torch.long)\n        lp0 = torch.log_softmax(z / T, dim=-1)\n        q0 = torch.zeros(B, dtype=torch.float64, device=device)\n        if rows:\n            q0.index_add_(0, rt, lp0.exp()[rt, ct])\n        app = pk_mask(mode, kk) & has if (h != 0.0) else torch.zeros(B, dtype=torch.bool, device=device)\n        if rows and bool(app.any()):\n            sel = app[rt]\n            z[rt[sel], ct[sel]] += h\n        lp = torch.log_softmax(z / T, dim=-1)\n        p = lp.exp()\n        q = torch.zeros(B, dtype=torch.float64, device=device)\n        if rows:\n            q.index_add_(0, rt, p[rt, ct])\n        kl = torch.where(app, (p * (lp - lp0)).nan_to_num(0.0).sum(-1), torch.zeros(B, dtype=torch.float64, device=device))\n        nxt = torch.multinomial(p, 1, generator=gen)\n        pk = torch.tensor([int(nxt[b, 0]) in set(cands[b]) for b in range(B)], device=device)\n        kk = torch.where(pk, kk + 1, torch.zeros_like(kk))\n        for b in range(B):\n            idx[b].push(int(nxt[b, 0]))\n        for k, v in ((\'toks\', nxt[:, 0]), (\'cand\', has), (\'pick\', pk), (\'U\', -lp1.gather(-1, nxt)[:, 0]), (\'q0\', q0), (\'q\', q), (\'kl\', kl), (\'app\', app)):\n            rec[k].append(v)\n        out = model(nxt, past_key_values=past, use_cache=True)\n        past, logits = out.past_key_values, out.logits[:, -1, :].float()\n    st = lambda v: torch.stack(v, 1).cpu().numpy()\n    dt = dict(toks=np.int32, cand=np.bool_, pick=np.bool_, U=np.float32, q0=np.float32, q=np.float32, kl=np.float32, app=np.bool_)\n    return {k: st(v).astype(dt[k]) for k, v in rec.items()}\n\n\ndef pk_jobs():\n    return [(name, hf, mode, h) for name, hf in PK_MODELS for mode, h in pk_configs()]\n\n\ndef pk_claim(outdir, tag, worker):\n    if os.path.exists(f\'{outdir}/{tag}.npz\') or os.path.exists(f\'{outdir}/locks/{tag}.failed\'):\n        return False\n    lk = f\'{outdir}/locks/{tag}.lock\'\n    if os.path.exists(lk):\n        try:\n            age = time.time() - json.load(open(lk))[\'t\']\n        except Exception:\n            age = 0.0\n        if age < PK_STALE:\n            return False\n    os.makedirs(f\'{outdir}/locks\', exist_ok=True)\n    json.dump(dict(worker=worker, host=socket.gethostname(), t=time.time()), open(lk, \'w\'))\n    time.sleep(20)\n    try:\n        return json.load(open(lk)).get(\'worker\') == worker\n    except Exception:\n        return False\n\n\ndef run_pork_job(name, hf_id, mode, h, docs, outdir, device, cache):\n    if cache.get(\'name\') != name:\n        cache.clear(); gc.collect(); torch.cuda.empty_cache()\n        tok, model = load_model(hf_id, device)\n        assert next(model.parameters()).dtype == torch.float32\n        valid, eos = valid_mask(tok, model, device)\n        pairs = [split_doc(tok, d[\'text\']) for d in docs]\n        use = [i for i, p in enumerate(pairs) if p is not None]\n        assert len({len(pairs[i][0]) for i in use}) == 1\n        cache.update(name=name, model=model, valid=valid, use=use, prompts=[pairs[i][0] for i in use])\n    model, valid, use, prompts = cache[\'model\'], cache[\'valid\'], cache[\'use\'], cache[\'prompts\']\n    nD = len(use)\n    chunks = [list(range(i, min(nD, i + PK_CHUNK))) for i in range(0, nD, PK_CHUNK)]\n    ta = time.time()\n    dt = dict(toks=np.int32, cand=np.bool_, pick=np.bool_, U=np.float32, q0=np.float32, q=np.float32, kl=np.float32, app=np.bool_)\n    R = {k: np.zeros((len(PK_TEMPS), PK_S, nD, CONT), v) for k, v in dt.items()}\n    for k, T in enumerate(PK_TEMPS):\n        for s in range(PK_S):\n            for c, ii in enumerate(chunks):\n                g = torch.Generator(device=device).manual_seed(pk_seed(k, s, c))\n                r = generate_pork(model, [prompts[i] for i in ii], T, h, mode, valid, g)\n                for kk in R:\n                    R[kk][k, s, ii] = r[kk]\n                torch.cuda.empty_cache()\n    tag = f\'{name}@{pk_tag(mode, h)}\'\n    path = f\'{outdir}/{tag}.npz\'\n    np.savez_compressed(path + \'.part.npz\', temps=np.array(PK_TEMPS), h=h, mode=mode, use=np.array(use), seconds=time.time() - ta, chunk=PK_CHUNK, **R)\n    os.replace(path + \'.part.npz\', path)\n    return time.time() - ta\n\n\ndef run_pork(worker, nworkers, docs, outdir, device, notify=print):\n    os.makedirs(outdir, exist_ok=True)\n    jobs = pk_jobs()\n    sh = (len(jobs) * worker) // nworkers\n    order = jobs[sh:] + jobs[:sh]\n    cache, n, t0 = {}, 0, time.time()\n    while True:\n        took = False\n        for name, hf, mode, h in order:\n            tag = f\'{name}@{pk_tag(mode, h)}\'\n            if pk_claim(outdir, tag, worker):\n                took = True\n                try:\n                    sec = run_pork_job(name, hf, mode, h, docs, outdir, device, cache)\n                    n += 1\n                    notify(f\'{tag} おわり（{sec:.0f} 秒、このランタイムで {n} 個目、通算 {time.time() - t0:.0f} 秒）\')\n                except Exception as e:\n                    import traceback\n                    txt = \'\'.join(traceback.format_exception(type(e), e, e.__traceback__))\n                    v = os.environ.get(\'HF_TOKEN\')\n                    if v:\n                        txt = txt.replace(v, \'***\')\n                    txt = txt.encode(\'utf-8\')[-3000:].decode(\'utf-8\', \'ignore\')\n                    json.dump(dict(worker=worker, t=time.time(), error=txt), open(f\'{outdir}/locks/{tag}.failed\', \'w\'), ensure_ascii=False)\n                    notify(f\'⚠️ {tag} で止まりました（この仕事は飛ばして、ほかを続けます）：\\n\' + txt)\n                    cache.clear(); gc.collect(); torch.cuda.empty_cache()\n                break\n        if not took:\n            break\n    cache.clear(); gc.collect(); torch.cuda.empty_cache()\n    left = []\n    for nm, _, mode, h in jobs:\n        tag = f\'{nm}@{pk_tag(mode, h)}\'\n        if not os.path.exists(f\'{outdir}/{tag}.npz\'):\n            left.append(tag + (\'（失敗）\' if os.path.exists(f\'{outdir}/locks/{tag}.failed\') else \'\'))\n    return n, left\n\n\n@torch.no_grad()\ndef self_test_pork(device):\n    """小さなランダムのモデルで：(1) 罰なし・\'all\' は generate_dsc2（一定の温度・一定の場）と同じトークン・q・q0。\n    (2) \'grow\'・\'melt\' は、記録した k の条件を満たす歩だけに罰がかかっている。(3) KL は罰をかけた歩だけ正、手で計算し直した値と合う。(4) 仕事の数と名前。"""\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    prompts = [[5, 6, 7, 8] * 4, [9, 10, 11, 12] * 4, [3, 4, 3, 4] * 4]\n    L = 60\n    for mode, h in ((\'none\', 0.0), (\'all\', -1.0)):\n        a = generate_pork(model, prompts, 0.6, h, mode, valid, torch.Generator(device=device).manual_seed(5), L)\n        b = generate_dsc2(model, prompts, np.full(L, 0.6), np.full(L, h), valid, torch.Generator(device=device).manual_seed(5))\n        assert (a[\'toks\'] == b[\'toks\']).all() and np.allclose(a[\'q\'], b[\'q\'], atol=1e-6) and np.allclose(a[\'q0\'], b[\'q0\'], atol=1e-6), mode\n    for mode in (\'grow\', \'melt\'):\n        a = generate_pork(model, prompts, 0.6, -2.0, mode, valid, torch.Generator(device=device).manual_seed(6), L)\n        k = np.zeros(a[\'pick\'].shape, int); run = np.zeros(a[\'pick\'].shape[0], int)\n        for t in range(L):\n            k[:, t] = run; run = np.where(a[\'pick\'][:, t], run + 1, 0)\n        want = a[\'cand\'] & (((k >= PK_KGROW[0]) & (k <= PK_KGROW[1])) if mode == \'grow\' else (k >= PK_KMELT))\n        assert (a[\'app\'] == want).all(), mode\n        assert (a[\'kl\'][~a[\'app\']] == 0).all() and (a[\'kl\'][a[\'app\']] >= -1e-7).all()\n        assert (a[\'q\'][a[\'app\']] <= a[\'q0\'][a[\'app\']] + 1e-6).all()\n    # KL を手で\n    a = generate_pork(model, prompts, 0.6, -1.0, \'all\', valid, torch.Generator(device=device).manual_seed(7), 20)\n    seq = a[\'toks\'][0].tolist()\n    lg = model(torch.tensor([prompts[0] + seq], device=device)).logits[0, len(prompts[0]) - 1: len(prompts[0]) - 1 + 20].double().masked_fill(~valid, float(\'-inf\'))\n    ci = CopyIndexN(prompts[0], B7_N)\n    for t in range(20):\n        c = list(ci.candidates())\n        z0 = lg[t].clone(); z1 = z0.clone()\n        if c:\n            z1[c] += -1.0\n        l0 = torch.log_softmax(z0 / 0.6, -1); l1 = torch.log_softmax(z1 / 0.6, -1)\n        klh = float((l1.exp() * (l1 - l0)).nan_to_num(0.0).sum()) if c else 0.0\n        assert abs(klh - float(a[\'kl\'][0, t])) < 2e-3, (t, klh, float(a[\'kl\'][0, t]))\n        ci.push(seq[t])\n    assert len(pk_jobs()) == 30 and len({(n, pk_tag(m, h)) for n, _, m, h in pk_jobs()}) == 30\n    return \'OK\'\n'
assert hashlib.sha256(SRCG.encode()).hexdigest() == '9cabe53a42dbdabbde8c52ed4c2bd84c100d3f1e86e3132bc287266edc24e393'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
exec(SRCD2)
exec(SRCG)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 仕事を取りに行って回す → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'), self_test_dsc2('cuda'), self_test_pork('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore/pork', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    N_DONE, LEFT = run_pork(0, 3, docs, '/content/drive/MyDrive/logit-thermo/A2_explore/pork', 'cuda', notify=lambda s: notify(s, tags='warning') if s.startswith('⚠') else notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(worker=0, done=N_DONE, left=LEFT, models=PK_MODELS, configs=pk_configs(), temps=PK_TEMPS, samples=PK_S, chunk=PK_CHUNK, seed=PK_SEED,
                   pork_sha256='9cabe53a42dbdabbde8c52ed4c2bd84c100d3f1e86e3132bc287266edc24e393', dsc2_sha256='cddae786903604183e46708a6606b482f720bba0b0df0a8df724440c9bde51b2', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_explore/pork' + '/run_meta_worker0.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 PROJECT_G の下見（ランタイム 1/3）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify(f'🎉 終わりました：探索 PROJECT_G の下見（ランタイム 1/3）。このランタイムで {N_DONE} 個。まだ残っている仕事：{len(LEFT)} 個' + (f'（{", ".join(LEFT)}）' if LEFT else '（全部そろいました）'), tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
